# 15.6 Wide & Deep

**中文**：FM 用因子化建模二阶交叉，但工业 CTR 还需要两种能力同时存在：**记忆（memorization）** 和 **泛化（generalization）**。Google 2016 年的 **Wide & Deep** 论文（用于 Google Play 应用推荐）给出经典答案——**用两条路并联**：一条"宽"的线性模型负责记住高频的、精确的特征组合规则；一条"深"的神经网络负责把稀疏特征嵌入稠密向量、泛化到没见过的组合。
**English**: FM models 2nd-order crosses via factorization, but industrial CTR needs two capabilities at once: **memorization** and **generalization**. Google's 2016 **Wide & Deep** paper (for Google Play app recommendation) gave the classic answer — **two parallel paths**: a "wide" linear model that memorizes frequent, exact feature-combination rules, and a "deep" network that embeds sparse features into dense vectors and generalizes to unseen combinations.

---

**中文**：
- **Wide（记忆）**：一个线性模型 $w^\top x$，输入包含原始稀疏特征 + **人工叉乘特征**（cross-product，如 `education=Master ∧ occupation=Prof`）。它能精确记住"这个组合历史上就是高点击"，**对高频精确规则极其有效**，但**无法泛化**到训练里没出现过的组合（权重为 0）。
- **Deep（泛化）**：把每个类别特征学成一个低维 embedding，拼起来过 MLP。embedding 让相似特征靠近，于是即使"某个具体组合没见过"，也能靠相邻 embedding **泛化**出合理预测。但深网有时会**过度泛化**——给本该是"例外/小众"的组合也推出平滑的高分。

**English**:
- **Wide (memorization)**: a linear model $w^\top x$ over raw sparse features + **manual cross-product features** (e.g. `education=Master ∧ occupation=Prof`). It memorizes "this exact combo historically clicked high" — extremely effective for frequent exact rules, but **cannot generalize** to combos never seen in training (weight = 0).
- **Deep (generalization)**: learn a low-dim embedding per categorical, concatenate, pass through an MLP. Embeddings place similar features nearby, so even an unseen combo gets a sensible prediction by **generalizing** from neighbors. But deep nets can **over-generalize** — smoothing a high score onto combos that should be rare exceptions.

$$\hat y = \sigma\Big(\underbrace{w_{\text{wide}}^\top[\,x,\ \phi(x)\,]}_{\text{记忆/memorize}} + \underbrace{w_{\text{deep}}^\top a^{(L)}}_{\text{泛化/generalize}} + b\Big)$$

**中文**：其中 $\phi(x)$ 是叉乘特征，$a^{(L)}$ 是深网最后一层激活。两条路的输出**相加**后再过 sigmoid，**联合训练**（一个损失同时更新两边）。
**English**: where $\phi(x)$ are cross features and $a^{(L)}$ is the deep net's last-layer activation. The two outputs are **summed**, then sigmoid, and **jointly trained** (one loss updates both sides).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 必考）**
> **中文**：**Wide = 记忆（线性 + 叉乘特征，精确但不泛化）；Deep = 泛化（embedding + MLP，泛化但可能过度）**。两者**联合训练**互补：deep 负责覆盖长尾/未见组合，wide 负责守住"明确规则"和"例外"。关键点：① 叉乘特征是人工的（wide 的工程负担）；② 联合训练 ≠ 各自训练再集成（梯度共享）；③ **不是所有数据都需要 wide**——当信号本身可泛化时，deep 单独就够，wide 增益有限甚至是噪声。这是后续 DeepFM 的动机：**用 FM 自动学交叉替代人工叉乘**。
> **English**: **Wide = memorization (linear + crosses, exact but no generalization); Deep = generalization (embeddings + MLP, generalizes but may over-smooth)**. **Jointly trained** to complement: deep covers the long tail / unseen combos, wide guards "explicit rules" and "exceptions." Key points: ① cross features are manual (wide's engineering burden); ② joint training ≠ train-separately-then-ensemble (shared gradients); ③ **not every dataset needs wide** — when the signal generalizes, deep alone suffices and wide adds little or noise. This motivates DeepFM next: **replace manual crosses with FM-learned crosses**.


## 实验一：合成数据证明"记忆与泛化缺一不可" / Synthetic: memorization & generalization need each other

**中文**：先用一个干净的合成实验把机制讲透。我们造两类信号：
**English**: First a clean synthetic experiment to nail the mechanism. We build two kinds of signal:

**中文**：
- **可泛化信号**：一个连续特征 $x$，点击率随 $x$ 平滑变化（$\sigma(2.5x)$）——这是 **deep 的主场**。
- **需要记忆的"例外规则"**：一个高基数类别 `rule_id`（600 个取值），其中随机 20% 的 id 是"例外"——只要命中就**必定点击**（覆盖 $x$）。这种"精确、稀疏、无规律"的规则正是 **wide 的主场**。

为了把机制隔离干净，我们**故意**让 deep 只看 $x$、wide 只看 `rule_id`：这样就能清楚看到"谁能学到什么"。

**English**:
- **Generalizable signal**: a continuous feature $x$ whose CTR varies smoothly ($\sigma(2.5x)$) — **deep's home turf**.
- **Memorization "exception rules"**: a high-cardinality categorical `rule_id` (600 values), of which a random 20% are "exceptions" — hitting one **forces a click** (overriding $x$). Such exact, sparse, irregular rules are **wide's home turf**.

To isolate the mechanism, we **deliberately** let deep see only $x$ and wide see only `rule_id`, making "who can learn what" crystal clear.


In [ ]:

# ============================================================
# 合成数据 + Wide / Deep / Wide&Deep（torch）/ synthetic + three models
# ============================================================
import numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

N, C = 24000, 600
x   = np.random.randn(N).astype(np.float32)                 # 连续可泛化特征 / generalizable feature
rid = np.random.randint(0, C, N)                            # 高基数类别 / high-card categorical
exc = np.zeros(C, bool); exc[np.random.choice(C,int(0.2*C),replace=False)]=True   # 20% 例外规则 / exception ids
y = (np.random.rand(N) < 1/(1+np.exp(-2.5*x))).astype(np.float32)   # 平滑信号采样 / smooth signal
y[exc[rid]] = 1.0                                           # 命中例外 id => 必点击 / forced click
te=slice(18000,N); yte=y[18000:]
X=torch.tensor(x[:,None]); RID=torch.tensor(rid); Y=torch.tensor(y)

def auc(yy,pp):
    o=np.argsort(pp); r=np.empty(len(pp)); r[o]=np.arange(len(pp))
    npos=yy.sum(); nneg=len(yy)-npos
    return (r[yy==1].sum()-npos*(npos-1)/2)/(npos*nneg)

class WideDeep(nn.Module):
    def __init__(s, mode):
        super().__init__(); s.mode=mode
        s.wide=nn.Embedding(C,1); nn.init.zeros_(s.wide.weight)             # wide: 每个 id 一个线性权重 / per-id weight
        s.deep=nn.Sequential(nn.Linear(1,32),nn.ReLU(),nn.Linear(32,1))     # deep: 只吃连续 x / consumes x only
        s.b=nn.Parameter(torch.zeros(1))
    def forward(s, xx, rr):
        o=s.b
        if s.mode in ("wide","wd"): o=o + s.wide(rr).squeeze(1)             # 记忆路 / memorization path
        if s.mode in ("deep","wd"): o=o + s.deep(xx).squeeze(1)            # 泛化路 / generalization path
        return o

def fit(mode, epochs=30, bs=256, lr=1e-2):
    torch.manual_seed(0); m=WideDeep(mode); opt=torch.optim.Adam(m.parameters(),lr=lr)
    lf=nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        perm=torch.randperm(18000)
        for i in range(0,18000,bs):
            idx=perm[i:i+bs]; opt.zero_grad(); lf(m(X[idx],RID[idx]),Y[idx]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): p=torch.sigmoid(m(X[te],RID[te])).numpy()
    return auc(yte,p), m

syn_auc={}
for mode in ["wide","deep","wd"]:
    syn_auc[mode],_=fit(mode); print(f"{mode:5}  test AUC = {syn_auc[mode]:.4f}")
print(f"\n例外规则占测试集正样本比例 / exceptions' share of test positives: {exc[rid[18000:]][yte==1].mean():.1%}")


**中文**：结果一目了然：**wide 单独 ≈ 0.67**（只抓住了例外规则、对平滑信号无能为力）；**deep 单独 ≈ 0.84**（学会了平滑信号、但把例外当噪声平滑掉了）；**Wide & Deep ≈ 0.92**——把两种能力**叠加**，显著超过任何单独一条路。这就是"记忆 + 泛化缺一不可"的最干净证明。下面可视化它们各自学到了什么。
**English**: The result is unambiguous: **wide alone ≈ 0.67** (captures the exception rules, helpless on the smooth signal); **deep alone ≈ 0.84** (learns the smooth signal but smooths the exceptions away as noise); **Wide & Deep ≈ 0.92** — **stacking** both capabilities clearly beats either path alone. This is the cleanest proof that "memorization + generalization need each other." Let's visualize what each learned.


In [ ]:

# ============================================================
# 可视化：各模型对"平滑信号"和"例外规则"的拟合 / what each model learned
# ============================================================
_,m_deep=fit("deep"); _,m_wide=fit("wide"); _,m_wd=fit("wd")
fig,ax=plt.subplots(1,3,figsize=(15,4.2))

# ① AUC 柱状 / AUC bars
names=["wide\n(记忆)","deep\n(泛化)","W&D\n(both)"]; vals=[syn_auc["wide"],syn_auc["deep"],syn_auc["wd"]]
ax[0].bar(names,vals,color=["#C44E52","#55A868","#4C72B0"]); ax[0].set_ylim(0.5,1.0)
for i,v in enumerate(vals): ax[0].text(i,v+0.01,f"{v:.3f}",ha="center")
ax[0].set_title("AUC：缺一不可 / neither alone suffices"); ax[0].set_ylabel("test AUC")

# ② 预测 CTR 随连续特征 x 的变化(取非例外样本)/ predicted CTR vs x on non-exception samples
xs=np.linspace(-3,3,60).astype(np.float32)
with torch.no_grad():
    # 用一个非例外的 rid 固定 / fix a non-exception id
    nonexc=int(np.where(~exc)[0][0]); rr=torch.full((60,),nonexc)
    pd_=torch.sigmoid(m_deep(torch.tensor(xs[:,None]),rr)).numpy()
    pw_=torch.sigmoid(m_wide(torch.tensor(xs[:,None]),rr)).numpy()
ax[1].plot(xs,1/(1+np.exp(-2.5*xs)),"k--",label="真实/true σ(2.5x)")
ax[1].plot(xs,pd_,label="deep",color="#55A868"); ax[1].plot(xs,pw_,label="wide",color="#C44E52")
ax[1].set_title("平滑信号：deep 学得到 / smooth signal: deep gets it"); ax[1].set_xlabel("x"); ax[1].set_ylabel("pred CTR"); ax[1].legend()

# ③ 例外 id 上的预测：wide 记住、deep 没记住 / on exception ids: wide remembers, deep doesn't
exc_ids=np.where(exc)[0][:200]
with torch.no_grad():
    xc=torch.zeros(len(exc_ids),1); ri=torch.tensor(exc_ids)
    pe_deep=torch.sigmoid(m_deep(xc,ri)).numpy(); pe_wide=torch.sigmoid(m_wide(xc,ri)).numpy()
ax[2].hist(pe_deep,bins=20,alpha=0.6,label="deep",color="#55A868")
ax[2].hist(pe_wide,bins=20,alpha=0.6,label="wide",color="#C44E52")
ax[2].axvline(1.0,ls="--",color="k"); ax[2].set_title("例外规则应预测≈1 / exceptions should be ≈1")
ax[2].set_xlabel("pred CTR on exception ids"); ax[2].legend()
plt.tight_layout(); plt.savefig("/tmp/rec06_syn.png",dpi=80); plt.show()
print("例外 id 上平均预测 / mean pred on exceptions:  wide=%.3f  deep=%.3f"%(pe_wide.mean(),pe_deep.mean()))


## 实验二：真实数据 Census/Adult / Real data — does Wide&Deep always win?

**中文**：合成实验证明了机制。现在上**真实数据**——经典的 **Census/Adult 收入数据集**（也是原版 Wide & Deep 教程用的数据）：根据年龄、学历、职业、婚姻、种族、性别、国籍等预测"年收入是否 > 50K"。这次三条路（wide / deep / wide&deep）的两侧都用**完整特征**（更接近工业实践），诚实地看 Wide&Deep 是否一定赢。
**English**: The synthetic proved the mechanism. Now **real data** — the classic **Census/Adult income dataset** (used by the original Wide & Deep tutorial): predict "income > 50K" from age, education, occupation, marital status, race, sex, country, etc. This time all three paths use **full features** on both sides (closer to industry), and we honestly check whether Wide&Deep always wins.


In [ ]:

# ============================================================
# Census/Adult 数据 + 特征工程 / Adult data + features
# ============================================================
import os, pandas as pd
R=os.path.expanduser("~/.cache/dsfs_recsys")
cols=["age","workclass","fnlwgt","education","edu_num","marital","occupation","relationship",
      "race","sex","capgain","caploss","hours","country","income"]
def load(fn,test=False):
    df=pd.read_csv(os.path.join(R,fn),names=cols,skipinitialspace=True,na_values="?",
                   skiprows=1 if test else 0).dropna().reset_index(drop=True)
    y=(df["income"].str.replace(".","",regex=False)==">50K").astype(np.float32).values
    return df,y
tr,ytr=load("adult.data"); ted,yte2=load("adult.test",test=True)
CAT=["workclass","education","marital","occupation","relationship","race","sex","country"]
CON=["age","edu_num","capgain","caploss","hours"]
vocab={c:{v:i+1 for i,v in enumerate(sorted(tr[c].unique()))} for c in CAT}    # 0 = 未登录/unk
nuniq=[max(vocab[c].values())+1 for c in CAT]
def cat_ids(df): return np.stack([[vocab[c].get(v,0) for v in df[c]] for c in CAT],1)
mu=tr[CON].mean(); sd=tr[CON].std()
def num(df): return ((df[CON]-mu)/sd).values.astype(np.float32)
# 宽侧特征：原始类别(各自偏移到全局下标) + 3 个人工叉乘 / wide: raw cats(offset) + 3 crosses
offs=np.cumsum([0]+nuniq)[:-1]; baseW=int(np.sum(nuniq))
def crs(df,a,b,B): return np.array([hash((u,v))%B for u,v in zip(df[a],df[b])])
def wide_feats(df,B=2000):
    raw=cat_ids(df)+offs
    c=[baseW+crs(df,"education","occupation",B), baseW+B+crs(df,"marital","relationship",B),
       baseW+2*B+crs(df,"occupation","sex",B)]
    return np.concatenate([raw]+[ci[:,None] for ci in c],1), baseW+3*B
Wtr,WB=wide_feats(tr); Wte,_=wide_feats(ted)
Ctr,Ntr=torch.tensor(cat_ids(tr)),torch.tensor(num(tr)); Wtr_t=torch.tensor(Wtr); Ytr=torch.tensor(ytr)
Cte,Nte=torch.tensor(cat_ids(ted)),torch.tensor(num(ted)); Wte_t=torch.tensor(Wte);
print(f"训练 {len(tr)}, 测试 {len(ted)}, 类别特征 {len(CAT)}, 宽侧特征维度 WB={WB}, 正例率 {ytr.mean():.1%}")


In [ ]:

# ============================================================
# 完整 Wide & Deep（embedding 深塔 + 叉乘宽塔）/ full Wide & Deep on Adult
# ============================================================
class WD2(nn.Module):
    def __init__(s, mode, emb=16, hid=(64,32)):
        super().__init__(); s.mode=mode
        s.wide=nn.EmbeddingBag(WB,1,mode="sum")                       # 宽塔：叉乘+原始特征求和 / wide
        s.embs=nn.ModuleList([nn.Embedding(n,emb) for n in nuniq])    # 深塔：每个类别一个 embedding / deep
        din=emb*len(CAT)+len(CON); L=[]
        for h in hid: L+=[nn.Linear(din,h),nn.ReLU(),nn.Dropout(0.1)]; din=h
        L+=[nn.Linear(din,1)]; s.deep=nn.Sequential(*L); s.b=nn.Parameter(torch.zeros(1))
    def forward(s,c,n,w):
        o=s.b
        if s.mode in ("wide","wd"): o=o+s.wide(w).squeeze(1)
        if s.mode in ("deep","wd"):
            e=torch.cat([emb(c[:,i]) for i,emb in enumerate(s.embs)],1)
            o=o+s.deep(torch.cat([e,n],1)).squeeze(1)
        return o
def fit2(mode, epochs=10, bs=512, lr=1e-3, seed=0):
    torch.manual_seed(seed); m=WD2(mode); opt=torch.optim.Adam(m.parameters(),lr=lr,weight_decay=1e-5)
    lf=nn.BCEWithLogitsLoss(); N=len(Ytr); curve=[]
    for ep in range(epochs):
        perm=torch.randperm(N)
        for i in range(0,N,bs):
            idx=perm[i:i+bs]; opt.zero_grad(); lf(m(Ctr[idx],Ntr[idx],Wtr_t[idx]),Ytr[idx]).backward(); opt.step()
        m.eval()
        with torch.no_grad(): p=torch.sigmoid(m(Cte,Nte,Wte_t)).numpy()
        curve.append(auc(yte2,p)); m.train()
    return curve
adult={}
for mode in ["wide","deep","wd"]:
    adult[mode]=fit2(mode); print(f"{mode:5}  test AUC = {adult[mode][-1]:.4f}")


In [ ]:

# 真实数据上的学习曲线 / learning curves on Adult
plt.figure(figsize=(6,4))
for mode,c in zip(["wide","deep","wd"],["#C44E52","#55A868","#4C72B0"]):
    plt.plot(range(1,len(adult[mode])+1),adult[mode],"o-",label=f"{mode} (final {adult[mode][-1]:.3f})",color=c)
plt.xlabel("epoch"); plt.ylabel("test AUC"); plt.title("Census/Adult: Wide vs Deep vs Wide&Deep")
plt.legend(); plt.tight_layout(); plt.savefig("/tmp/rec06_adult.png",dpi=80); plt.show()


**中文**：诚实结果——在 Census/Adult 上，**deep ≈ Wide&Deep ≫ wide**，而且 **Wide&Deep 并没有明显超过 deep 单独**（有时还略低）。把合成实验和真实实验放在一起，得到本节最重要的两条洞察：
**English**: Honest result — on Census/Adult, **deep ≈ Wide&Deep ≫ wide**, and **Wide&Deep does not clearly beat deep alone** (sometimes slightly lower). Putting the synthetic and real experiments together yields this section's two key insights:

**中文**：
1. **机制是真的，但增益取决于数据**：合成数据里存在"deep 学不到、wide 才能记住"的稀疏例外规则，所以 Wide&Deep 大胜；而 Adult 的信号高度**可泛化**（学历↑、资本收益↑ → 收入↑，这种平滑趋势 deep 的 embedding+MLP 本就擅长），几乎没有"非记不可的例外"，于是 wide 加不进新信息，甚至引入噪声。
2. **这正是 Google 论文的语境**：Wide&Deep 在 Google Play 大放异彩，是因为 app 推荐里**存在大量精确的小众规则**（装了 App A 的人**就是**会点 App B），这种 memorization 深网容易过度泛化掉。**没有这种规则的数据，deep 单独就够。**

**English**:
1. **The mechanism is real, but the gain is data-dependent**: the synthetic data contains sparse exception rules "deep can't learn, only wide can memorize," so Wide&Deep wins big; Adult's signal is highly **generalizable** (more education ↑, capital gain ↑ → income ↑ — smooth trends that deep's embedding+MLP already handle), with almost no "must-memorize exceptions," so wide adds no new signal and even injects noise.
2. **This is exactly the Google paper's context**: Wide&Deep shone on Google Play because app recommendation has **many exact niche rules** (people who installed App A *do* click App B) — memorization that deep nets tend to over-generalize away. **Without such rules, deep alone is enough.**

> 💼 **实战视角 / Practical angle**
> **中文**：① **wide 的痛点是人工叉乘特征**——要工程师猜哪些组合重要。下一节 **DeepFM** 用 FM 自动学交叉来替代它，省掉人工。② 上线一个新架构前，**永远先验证 deep-only 基线**——很多"花哨结构"在你的数据上并不比 deep 强。③ 面试金句：*"Wide 记忆、Deep 泛化、联合训练互补；但 wide 的价值取决于数据里是否存在稀疏的精确规则。"*
> **English**: ① **wide's pain is manual cross features** — engineers must guess which combos matter. Next, **DeepFM** replaces this with FM-learned crosses. ② Before shipping a fancy architecture, **always check the deep-only baseline** — many "fancy structures" don't beat plain deep on your data. ③ Interview line: *"Wide memorizes, Deep generalizes, joint training complements; but wide's value depends on whether sparse exact rules exist in your data."*

---
### 小结 / Summary
- **中文**：Wide&Deep = 线性记忆塔(原始+叉乘特征) ∥ 深度泛化塔(embedding+MLP)，输出相加联合训练。
- **English**: Wide&Deep = linear memorization tower (raw + cross features) ∥ deep generalization tower (embeddings + MLP), summed and jointly trained.
- **中文**：合成例外规则数据上"缺一不可"(0.92>0.84>0.67)；真实 Adult 上 deep≈wd≫wide——增益看数据是否有稀疏例外。
- **English**: On synthetic exception-rule data "neither alone suffices" (0.92>0.84>0.67); on real Adult, deep≈wd≫wide — the gain depends on whether sparse exceptions exist.
- **中文**：wide 靠人工叉乘是其软肋，引出下一节 DeepFM 用 FM 自动学交叉。
- **English**: wide's manual crosses are its weakness, motivating DeepFM's FM-learned crosses next.
